# Manually review the 265 SinSpeech-flagged rows

`qa_flagged_for_review.parquet` (built alongside this notebook) embeds the WAV audio
for every row `qa_flagged_ids.csv` listed -- the 265 utterances SinSpeech's
reviewers deleted after listening, which `apply_sinspeech_qa_openslr.ipynb` kept
in `utt_spk_text_qa.tsv` rather than dropping automatically (per instruction: no
row gets removed without a human decision).

Rows are pre-sorted by `category` (see the breakdown below) so the fast, obvious
calls come first:

- **A_number_fragment** (45) -- a bare classifier word (`ක`/`ක්`/`කට`/`කින්`) where
  a numeral should be, e.g. `විනාඩි ක විතරත් ගියා`
- **B_duplicated_word** (3) -- stutter/scrape artifact, e.g. `සහ සහ`
- **C_garbled_conjunct** (10) -- vowel sign before the conjunct instead of after
  (`පි්‍රය` instead of `ප්‍රිය`) -- a real typo, not a joiner-placement issue
- **D_stray_period** (54) -- likely the same missing-numeral defect as A, e.g.
  `වේලාව රාත්‍රි .ට පමණ ඇත.`
- **E_initials** (7) -- period-separated initials, e.g. `ඒ.ඒස්.අනුර දිසානායක`
- **F_other** (146) -- everything else: colloquial/slang, truncated sentences,
  needs an actual listen to judge

**Workflow:** run the review-loop cell, listen to each clip, type a one-letter
decision. Every decision is written to `qa_manual_review_log.csv` immediately --
close the notebook anytime and rerun the loop cell later to pick up where you left
off (already-reviewed ids are skipped automatically).

In [1]:
import datetime
import io
import os

import pandas as pd
import soundfile as sf
from IPython.display import Audio, clear_output, display

CLEAN_DIR = "../../../../data/raw/openslr_52/processed/openslr_52_clean"
REVIEW_PARQUET = os.path.join(CLEAN_DIR, "qa_flagged_for_review.parquet")
REVIEW_LOG = os.path.join(CLEAN_DIR, "qa_manual_review_log.csv")
QA_TSV = os.path.join(CLEAN_DIR, "utt_spk_text_qa.tsv")

## Load rows + resume state

Safe to rerun this cell anytime -- it just re-reads whatever's on disk.

In [2]:
df = pd.read_parquet(REVIEW_PARQUET)
print(f"{len(df)} rows to review")
print(df["category"].value_counts())

if os.path.exists(REVIEW_LOG):
    log = pd.read_csv(REVIEW_LOG, dtype=str, keep_default_na=False)
else:
    log = pd.DataFrame(columns=["file_id", "category", "original_text", "final_text", "action", "reviewed_at"])

reviewed_ids = set(log["file_id"])
remaining = df[~df["file_id"].isin(reviewed_ids)].reset_index(drop=True)
print(f"\nalready reviewed: {len(reviewed_ids)} / {len(df)}")
print(f"remaining: {len(remaining)}")

265 rows to review
category
F_other               146
D_stray_period         54
A_number_fragment      45
C_garbled_conjunct     10
E_initials              7
B_duplicated_word       3
Name: count, dtype: int64

already reviewed: 16 / 265
remaining: 249


## Optional: focus on one category at a time

Leave `CATEGORY_FILTER = None` to go through everything in the pre-sorted order,
or set it (e.g. `"A_number_fragment"`) to only queue up that category this
session.

In [5]:
CATEGORY_FILTER = 'E_initials'   # e.g. "A_number_fragment", "F_other", ... or None for all

queue = remaining if CATEGORY_FILTER is None else remaining[remaining["category"] == CATEGORY_FILTER].reset_index(drop=True)
print(f"queued {len(queue)} rows" + (f" (category={CATEGORY_FILTER})" if CATEGORY_FILTER else ""))

queued 7 rows (category=E_initials)


## Review loop

For each clip: listens, shows the transcript, then asks for a decision --

- `k` -- keep text exactly as-is (SinSpeech's concern doesn't apply / is fine)
- `e` -- edit: type a corrected transcript
- `d` -- delete: confirm this row should be dropped from training data
- `s` -- skip for now, ask again next session
- `q` -- stop reviewing, save and exit (safe -- nothing is lost)

Saves to `qa_manual_review_log.csv` after **every single decision**, so an
interrupted kernel never loses more than the row in progress.

In [ ]:
def save_log():
    log.to_csv(REVIEW_LOG, index=False)


for _, row in queue.iterrows():
    clear_output(wait=True)
    print(f"file_id: {row['file_id']}   category: {row['category']}")
    print(f"text:    {row['text']!r}")

    data, sr = sf.read(io.BytesIO(row["audio"]))
    display(Audio(data=data, rate=sr, autoplay=True))

    action = 'e'

    if action == "q":
        print("stopped -- progress saved, rerun this cell to resume")
        break
    if action == "s":
        continue

    if action == "e":
        new_text = input(f"corrected text (was: {row['text']!r}): ").strip()
        final_text, act = new_text, "edited"
    elif action == "d":
        final_text, act = row["text"], "delete"
    else:
        final_text, act = row["text"], "keep"

    log.loc[len(log)] = [
        row["file_id"], row["category"], row["text"], final_text, act,
        datetime.datetime.now().isoformat(timespec="seconds"),
    ]
    save_log()

print(f"\nsession done. total reviewed so far: {len(log)} / {len(df)}")

file_id: cff3924d14   category: E_initials
text:    'උප පොලිස් පරීක්ෂකවරයකු ලෙසට ඇමැති ආරක්ෂක අංශයේ සේවය කරන ඒ.ටී.'


## Progress summary

In [4]:
if len(log):
    print(log["action"].value_counts())
    print(f"\n{len(df) - len(log)} rows still unreviewed")
else:
    print("no rows reviewed yet")

no rows reviewed yet


## Apply reviewed decisions to a new TSV

**Not run automatically -- read before running.** Only acts on rows that have an
explicit human decision in `qa_manual_review_log.csv`; anything not yet reviewed is
left untouched (kept, with its original text) so this is safe to run after a
partial review session too. Writes a **new** `utt_spk_text_qa_reviewed.tsv` --
does not overwrite `utt_spk_text_qa.tsv`.

In [ ]:
APPLY_RESULTS = False  # flip to True when ready to fold qa_manual_review_log.csv into a new TSV

if APPLY_RESULTS:
    qa = pd.read_csv(QA_TSV, sep="\t", dtype=str, keep_default_na=False)

    log_by_id = log.set_index("file_id")
    delete_ids = set(log_by_id[log_by_id["action"] == "delete"].index)
    edit_map = log_by_id[log_by_id["action"] == "edited"]["final_text"].to_dict()

    before = len(qa)
    qa = qa[~qa["file_id"].isin(delete_ids)].reset_index(drop=True)
    qa["text"] = qa.apply(lambda r: edit_map.get(r["file_id"], r["text"]), axis=1)
    qa["text_len"] = qa["text"].str.len()

    out_path = os.path.join(CLEAN_DIR, "utt_spk_text_qa_reviewed.tsv")
    qa.to_csv(out_path, sep="\t", index=False)
    print(f"dropped {before - len(qa)} rows (manually confirmed delete)")
    print(f"edited {len(edit_map)} rows")
    print(f"wrote {len(qa)} rows -> {out_path}")
else:
    print("APPLY_RESULTS is False -- skipped. Set to True once you've reviewed what you want to apply.")